In [0]:
-- Databricks notebook source
-- ============================================================================
-- Silver cleaning: Happy Family Homestay bookings
--
-- Source : happy_family_homestay.bronze.homestay (raw strings)
-- Target : happy_family_homestay.silver.homestay_clean
-- Grain  : one row per booking (id)
-- Load   : incremental upsert (MERGE on id). Safe to re-run.
--
-- Business rules applied here:
--   * Text is trimmed and lowercased (guest_name, email, city).
--   * Timestamps and dates are parsed from two formats: M/d/yyyy and yyyy-MM-dd.
--   * Rows without a valid check_in date are dropped.
--   * jumlah_kamar is free text (e.g. "2 Kamar, 1 Rumah"); the number in front of
--     each accommodation type is extracted into its own column.
--   * total_rooms_booked counts room equivalents: rumah = 5, paviliun = 2, kamar = 1.
--   * standard_total_revenue = standard nightly rates x stay_nights:
--       kamar 200,000 | rumah 1,300,000 | paviliun 400,000
--   * total_revenue = manual total from the form if filled in, else the standard total.
-- ============================================================================

-- COMMAND ----------

-- 1. Create the schema and the target table (no-op if they already exist)

CREATE SCHEMA IF NOT EXISTS happy_family_homestay.silver
COMMENT 'Silver layer: cleaned and typed data';

CREATE TABLE IF NOT EXISTS happy_family_homestay.silver.homestay_clean (
    id                      STRING        COMMENT 'Booking id: "BK-" + first 12 characters of md5(created_at, guest_name)',
    created_at              TIMESTAMP     COMMENT 'Form submission time',
    guest_name              VARCHAR(50)   COMMENT 'Guest full name (trimmed, lowercase)',
    phone_number            CHAR(14)      COMMENT 'Guest mobile number',
    email                   VARCHAR(50)   COMMENT 'Guest email address (trimmed, lowercase)',
    check_in                DATE          COMMENT 'Check-in date',
    check_out               DATE          COMMENT 'Check-out date',
    stay_nights             SMALLINT      COMMENT 'Nights stayed: check_out - check_in',
    guest_count             SMALLINT      COMMENT 'Number of guests',
    city                    CHAR(30)      COMMENT 'Guest home city (trimmed, lowercase)',
    visiting_purpose        CHAR(30)      COMMENT 'Purpose of the trip',
    acquisition_channel     CHAR(30)      COMMENT 'How the guest heard about the homestay',
    kamar_count             SMALLINT      COMMENT 'Number of Kamar (rooms) booked',
    rumah_count             SMALLINT      COMMENT 'Number of Rumah (houses) booked',
    paviliun_count          SMALLINT      COMMENT 'Number of Paviliun (pavilions) booked',
    total_rooms_booked      SMALLINT      COMMENT 'Room equivalents: rumah x 5 + paviliun x 2 + kamar',
    nightly_rate            DOUBLE        COMMENT 'total_revenue / stay_nights',
    manual_total_price      DOUBLE        COMMENT 'Total entered manually in the form (NULL if empty)',
    standard_total_revenue  DOUBLE        COMMENT 'Standard nightly rates x stay_nights',
    total_revenue           DOUBLE        COMMENT 'manual_total_price if present, else standard_total_revenue'
)
USING DELTA
COMMENT 'Cleaned homestay bookings, one row per booking';

-- COMMAND ----------

-- 2. Clean and enrich the bronze data in three steps:
--    parsed_data     -> parse and normalise the raw text columns
--    calculated_base -> nights, room equivalents, standard revenue
--    priced          -> final revenue (manual price wins over standard price)

CREATE OR REPLACE TEMP VIEW tv_homestay_clean AS
WITH parsed_data AS (
    SELECT
        COALESCE(
            try_to_timestamp(timestamp, 'M/d/yyyy H:mm:ss'),
            try_to_timestamp(timestamp, 'yyyy-MM-dd H:mm:ss')
        )                                                       AS created_at,
        LOWER(TRIM(nama_lengkap))                               AS guest_name,
        nomor_hp                                                AS phone_number,
        LOWER(TRIM(e_mail))                                     AS email,
        COALESCE(
            try_to_date(tanggal_check_in, 'M/d/yyyy'),
            try_to_date(tanggal_check_in, 'yyyy-MM-dd')
        )                                                       AS check_in,
        COALESCE(
            try_to_date(tanggal_check_out, 'M/d/yyyy'),
            try_to_date(tanggal_check_out, 'yyyy-MM-dd')
        )                                                       AS check_out,
        jumlah_tamu                                             AS guest_count,
        LOWER(TRIM(kota))                                       AS city,
        tujuan_perjalanan_wisata                                AS visiting_purpose,
        dari_mana_anda_mendengar_tentang_happy_family_homestay  AS acquisition_channel,
        -- Quantity in front of each accommodation type, 0 if the type is not booked
        COALESCE(NULLIF(REGEXP_EXTRACT(jumlah_kamar, '(\\d+)\\s*Kamar',    1), ''), 0) AS kamar_count,
        COALESCE(NULLIF(REGEXP_EXTRACT(jumlah_kamar, '(\\d+)\\s*Rumah',    1), ''), 0) AS rumah_count,
        COALESCE(NULLIF(REGEXP_EXTRACT(jumlah_kamar, '(\\d+)\\s*Paviliun', 1), ''), 0) AS paviliun_count,
        NULLIF(total_bayar, '')                                 AS manual_total_price
    FROM happy_family_homestay.bronze.homestay
),

calculated_base AS (
    SELECT
        *,
        DATEDIFF(check_out, check_in)                           AS stay_nights,
        -- Room equivalents: 1 rumah = 5 rooms, 1 paviliun = 2 rooms, 1 kamar = 1 room
        (rumah_count * 5 + paviliun_count * 2 + kamar_count)    AS total_rooms_booked,
        -- Standard nightly rates: kamar 200,000 | rumah 1,300,000 | paviliun 400,000
        (kamar_count * 200000 + rumah_count * 1300000 + paviliun_count * 400000)
            * stay_nights                                       AS standard_total_revenue
    FROM parsed_data
),

priced AS (
    SELECT
        *,
        COALESCE(manual_total_price, standard_total_revenue)    AS total_revenue
    FROM calculated_base
)

SELECT
    CONCAT('BK-', SUBSTRING(md5(CONCAT(created_at, guest_name)), 1, 12)) AS id,
    created_at,
    guest_name,
    phone_number,
    email,
    check_in,
    check_out,
    stay_nights,
    guest_count,
    city,
    visiting_purpose,
    acquisition_channel,
    kamar_count,
    rumah_count,
    paviliun_count,
    total_rooms_booked,
    total_revenue / NULLIF(stay_nights, 0)                      AS nightly_rate,
    manual_total_price,
    standard_total_revenue,
    total_revenue
FROM priced
WHERE check_in IS NOT NULL;

-- COMMAND ----------

-- 3. Upsert into silver: update existing bookings, insert new ones

MERGE INTO happy_family_homestay.silver.homestay_clean AS t
USING tv_homestay_clean AS s
    ON t.id = s.id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *;